In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import torch
import sys
sys.path.append("/data2/saikiran.tedla/hdrvideo/diff")


from PIL import Image
from diffsynth import load_state_dict
from diffsynth.pipelines.wan_video_new import WanVideoPipeline, ModelConfig

from diffsynth.trainers.stuttgart_dataset import StuttgartDataset
from utils import output_hdr_video, process_bracketed_video
from utils import average_frame_psnr



/data2/saikiran.tedla/anyedit/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
pipe = WanVideoPipeline.from_pretrained(
    torch_dtype=torch.bfloat16,
    device="cuda",
    model_configs=[
        ModelConfig(model_id="Wan-AI/Wan2.2-TI2V-5B", origin_file_pattern="models_t5_umt5-xxl-enc-bf16.pth", offload_device="cpu", skip_download=True),
        ModelConfig(model_id="Wan-AI/Wan2.2-TI2V-5B", origin_file_pattern="diffusion_pytorch_model*.safetensors", offload_device="cpu", skip_download=True),
        ModelConfig(model_id="Wan-AI/Wan2.2-TI2V-5B", origin_file_pattern="Wan2.2_VAE.pth", offload_device="cpu", skip_download=True),
    ],
)

dataset = StuttgartDataset(
    base_path="/data2/saikiran.tedla/hdrvideo/diff/data/stuttgart/carousel_fireworks_02",
    repeat=1,
    main_data_operator=StuttgartDataset.default_video_operator(
        base_path="/data2/saikiran.tedla/hdrvideo/diff/data/stuttgart/carousel_fireworks_02",
        max_pixels=1280*720,
        height=480,
        width=832,
        height_division_factor=16,
        width_division_factor=16,
        num_frames=15,
        time_division_factor=4,
        time_division_remainder=1,
    ),
    mode = "hdr_and_brackets"
)

# ensure the VAE is actually on the GPU in the right dtype
pipe.vae = pipe.vae.to(device=pipe.device)
pipe.vae.eval()  # optional but good practice

device = pipe.device

Loading models from: ./models/Wan-AI/Wan2.2-TI2V-5B/models_t5_umt5-xxl-enc-bf16.pth


    model_name: wan_video_text_encoder model_class: WanTextEncoder
    The following models are loaded: ['wan_video_text_encoder'].
Loading models from: ['./models/Wan-AI/Wan2.2-TI2V-5B/diffusion_pytorch_model-00003-of-00003-bf16.safetensors', './models/Wan-AI/Wan2.2-TI2V-5B/diffusion_pytorch_model-00002-of-00003-bf16.safetensors', './models/Wan-AI/Wan2.2-TI2V-5B/diffusion_pytorch_model-00001-of-00003-bf16.safetensors']
    model_name: wan_video_dit model_class: WanModel
        This model is initialized with extra kwargs: {'has_image_input': False, 'patch_size': [1, 2, 2], 'in_dim': 48, 'dim': 3072, 'ffn_dim': 14336, 'freq_dim': 256, 'text_dim': 4096, 'out_dim': 48, 'num_heads': 24, 'num_layers': 30, 'eps': 1e-06, 'seperated_timestep': True, 'require_clip_embedding': False, 'require_vae_embedding': False, 'fuse_vae_embedding_in_latents': True}
    The following models are loaded: ['wan_video_dit'].
Loading models from: ./models/Wan-AI/Wan2.2-TI2V-5B/Wan2.2_VAE.pth
    model_name: wan_

In [ ]:
# finetune_vae_decoder.py
# Fine-tunes ONLY pipe.vae.model.decoder on bracket->HDR loss.

import os
import copy
import math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# ----------------------------
# Config
# ----------------------------
# Assumptions:
# - `pipe` and `dataset` and `process_bracketed_video` exist in scope or are imported below.
# - `average_frame_psnr` is for logging only (optional).

NUM_EPOCHS     = 5
BATCH_SIZE     = 1
LR             = 1e-5
WEIGHT_DECAY   = 0.01
BETAS          = (0.9, 0.999)
GRAD_CLIP_NORM = 1.0
EXPOSURE_COUNTS = (5, 5, 5)  # (normal, low, high)
USE_TILED      = False       # set True if your VAE supports tiled encode/decode
SAVE_PATH      = "vae_decoder_finetuned.pt"
LOG_EVERY      = 50

# If you import from elsewhere, uncomment:
# from your_module import pipe, dataset, process_bracketed_video, average_frame_psnr

# ----------------------------
# Utilities
# ----------------------------
def pil_list_to_video_tensor(pil_frames, *, device, dtype=torch.bfloat16):
    """
    Convert a list of PIL images -> (1, T, C, H, W) in [0, 1].
    """
    frames = []
    for img in pil_frames:
        arr = np.asarray(img, dtype=np.uint8)           # (H, W, C), 0..255
        t = torch.from_numpy(arr).permute(2, 0, 1)      # (C, H, W)
        frames.append(t)
    video = torch.stack(frames, dim=0).float() / 255.0  # (T, C, H, W), 0..1
    return video.unsqueeze(0).to(device=device, dtype=dtype, non_blocking=True)

def to_vae_space(video_b_t_c_h_w):
    """
    Scale [0,1] -> [-1,1] and permute (B,T,C,H,W) -> (B,C,T,H,W) for VAE.
    """
    x = video_b_t_c_h_w * 2.0 - 1.0
    return x.permute(0, 2, 1, 3, 4).contiguous()

def from_vae_space(video_b_c_t_h_w):
    """
    Permute (B,C,T,H,W) -> (B,T,C,H,W) and scale [-1,1] -> [0,1].
    """
    x = video_b_c_t_h_w.permute(0, 2, 1, 3, 4)
    x = (x + 1.0) / 2.0
    return x.clamp_(0.0, 1.0)

def split_brackets(video_b_t_c_h_w, *, counts=(5, 5, 5)):
    """
    Split along T into exposure groups. Returns list of (B,T_i,C,H,W).
    """
    B, T, C, H, W = video_b_t_c_h_w.shape
    n0, n1, n2 = counts
    assert n0 + n1 + n2 == T, f"T={T} does not match counts {counts}"
    return [
        video_b_t_c_h_w[:, 0:n0],
        video_b_t_c_h_w[:, n0:n0 + n1],
        video_b_t_c_h_w[:, n0 + n1:n0 + n1 + n2],
    ]

def psnr_from_mse(mse: float, data_range: float = 1.0) -> float:
    if mse <= 0:
        return float("inf")
    return 20.0 * math.log10(data_range) - 10.0 * math.log10(mse)

# ----------------------------
# Prep modules
# ----------------------------
device = pipe.device if hasattr(pipe, "device") else torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype  = torch.bfloat16  # match your earlier setup
autocast_enabled = True #(device.type == "cuda")

# Validate decoder path
assert hasattr(pipe, "vae") and hasattr(pipe.vae, "model") and hasattr(pipe.vae.model, "decoder"), \
    "Expected decoder at pipe.vae.model.decoder"
decoder = pipe.vae.model.decoder

# Freeze everything except decoder
pipe.vae.requires_grad_(False)
decoder.requires_grad_(True)

decoder_params = list(decoder.parameters())
optimizer = torch.optim.AdamW(decoder_params, lr=LR, weight_decay=WEIGHT_DECAY, betas=BETAS)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS * max(1, len(dataset)//BATCH_SIZE))

# Optional: save initial decoder state to revert if needed
init_decoder_state = copy.deepcopy(decoder.state_dict())

# Data loader
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

# Loss (MSE = PSNR surrogate). You can swap to Charbonnier/SSIM/LPIPS if differentiable and available.
crit = nn.MSELoss()

# ----------------------------
# Train loop
# ----------------------------
pipe.vae.train()
global_step = 0

for epoch in range(NUM_EPOCHS):
    for it, data_item in enumerate(loader):
        # Prepare bracketed LDR and HDR target on GPU (stay on GPU for autograd)
        # DataLoader may wrap dicts in lists/tuples if collate_fn is default.
        # Handle both single-sample dict or batched dict-of-lists.
        video_input = data_item["video"] if isinstance(data_item["video"], (list, tuple)) else data_item["video"]
        hdr_np      = data_item["hdr_video"][0] if torch.is_tensor(data_item["hdr_video"]) else data_item["hdr_video"]
        video_input = torch.concatenate(video_input, dim=0) if isinstance(video_input, (list, tuple)) else video_input
        bracket_video = pil_list_to_video_tensor(video_input, device=device, dtype=dtype)  # (1,T,C,H,W)

        #cut to patch
        bracket_video = bracket_video[:, :,:, 0:96, 0:96]
        hdr_np = hdr_np[:, 0:96, 0:96, :]
        print(bracket_video.shape, hdr_np.shape)
        


        hdr_target = hdr_np if torch.is_tensor(hdr_np) else torch.from_numpy(hdr_np)  # (T,H,W,C)
        hdr_target = hdr_target.permute(0, 3, 1, 2)          # (T,C,H,W)
        hdr_target = hdr_target.unsqueeze(0).to(dtype=dtype) # (1,T,C,H,W)
        #put targets on device
        hdr_target = hdr_target.to(device=device, non_blocking=True)

        # Split exposures
        normal_exp, low_exp, high_exp = split_brackets(bracket_video, counts=EXPOSURE_COUNTS)

        # Encode (no grad)
        with torch.no_grad():
            lat_n = pipe.vae.encode(to_vae_space(normal_exp), device=device, tiled=USE_TILED)
            lat_l = pipe.vae.encode(to_vae_space(low_exp),    device=device, tiled=USE_TILED)
            lat_h = pipe.vae.encode(to_vae_space(high_exp),   device=device, tiled=USE_TILED)
            lat_n = lat_n.to(dtype=pipe.torch_dtype, device=device, non_blocking=True)
            lat_l = lat_l.to(dtype=pipe.torch_dtype, device=device, non_blocking=True)
            lat_h = lat_h.to(dtype=pipe.torch_dtype, device=device, non_blocking=True)

        # Decode (with grad) — hits pipe.vae.model.decoder
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=autocast_enabled):
            dec_n = from_vae_space(pipe.vae.decode(lat_n, device=device, tiled=USE_TILED))  # (1,Tn,C,H,W)
            dec_l = from_vae_space(pipe.vae.decode(lat_l, device=device, tiled=USE_TILED))
            dec_h = from_vae_space(pipe.vae.decode(lat_h, device=device, tiled=USE_TILED))

            decoded_combined = torch.cat([dec_n, dec_l, dec_h], dim=1)  # (1,T,C,H,W)

            # IMPORTANT: process_bracketed_video must be differentiable and on the same device
            hdr_pred = process_bracketed_video(decoded_combined[0])      # (T,C,H,W)
            hdr_pred = hdr_pred.unsqueeze(0)                             # (1,T,C,H,W)

            # Match dtype for loss
            hdr_pred   = hdr_pred.to(dtype=dtype)
            hdr_target = hdr_target.to(dtype=dtype)

            loss = crit(hdr_pred, hdr_target)
            print("Loss:", loss.item())

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(decoder_params, GRAD_CLIP_NORM)
        optimizer.step()
        scheduler.step()

        # Logging
        if (global_step + 1) % LOG_EVERY == 0:
            with torch.no_grad():
                mse = (hdr_pred.float() - hdr_target.float()).pow(2).mean().item()
                psnr = psnr_from_mse(mse, data_range=1.0)  # assuming [0,1]
            print(f"[{epoch+1}/{NUM_EPOCHS}] step {global_step+1} | loss {loss.item():.6f} | psnr ≈ {psnr:.2f} dB")

        global_step += 1

# ----------------------------
# Save
# ----------------------------
torch.save(decoder.state_dict(), SAVE_PATH)
print(f"Saved fine-tuned decoder to {SAVE_PATH}")

# ----------------------------
# (Optional) Quick eval on one sample with PSNR vs GT HDR
# ----------------------------
try:
    data0 = dataset[0]
    bvid = pil_list_to_video_tensor(data0["video"], device=device, dtype=dtype)
    n,l,h = split_brackets(bvid, counts=EXPOSURE_COUNTS)
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=autocast_enabled):
        dn = from_vae_space(pipe.vae.decode(pipe.vae.encode(to_vae_space(n), device=device, tiled=USE_TILED), device=device, tiled=USE_TILED))
        dl = from_vae_space(pipe.vae.decode(pipe.vae.encode(to_vae_space(l), device=device, tiled=USE_TILED), device=device, tiled=USE_TILED))
        dh = from_vae_space(pipe.vae.decode(pipe.vae.encode(to_vae_space(h), device=device, tiled=USE_TILED), device=device, tiled=USE_TILED))
        dcomb = torch.cat([dn, dl, dh], dim=1)
        hdr_eval = process_bracketed_video(dcomb[0]).unsqueeze(0).to(dtype=torch.float32)

    hdr_gt = torch.from_numpy(data0["hdr_video"]).permute(0,3,1,2).unsqueeze(0).to(dtype=torch.float32)
    mse = (hdr_eval.cpu() - hdr_gt).pow(2).mean().item()
    psnr = psnr_from_mse(mse, data_range=1.0)
    print(f"[Eval] PSNR of bracket→HDR after finetune: {psnr:.2f} dB")
except Exception as e:
    print(f"[Eval skipped] {e}")


torch.Size([1, 15, 3, 96, 96]) torch.Size([5, 96, 96, 3])
Loss: 1.0566044466031599e-06
torch.Size([1, 15, 3, 96, 96]) torch.Size([5, 96, 96, 3])
Loss: 0.018435535952448845
torch.Size([1, 15, 3, 96, 96]) torch.Size([5, 96, 96, 3])
Loss: 1.95200186681177e-06
torch.Size([1, 15, 3, 96, 96]) torch.Size([5, 96, 96, 3])
Loss: 1.2173638879175996e-06
torch.Size([1, 15, 3, 96, 96]) torch.Size([5, 96, 96, 3])
Loss: 1.2611055808520177e-06
torch.Size([1, 15, 3, 96, 96]) torch.Size([5, 96, 96, 3])
Loss: 0.00031069506076164544
torch.Size([1, 15, 3, 96, 96]) torch.Size([5, 96, 96, 3])
Loss: 0.10290341824293137
torch.Size([1, 15, 3, 96, 96]) torch.Size([5, 96, 96, 3])
Loss: 1.1086962103945552e-06
torch.Size([1, 15, 3, 96, 96]) torch.Size([5, 96, 96, 3])
Loss: 1.8979986862177611e-06
torch.Size([1, 15, 3, 96, 96]) torch.Size([5, 96, 96, 3])
Loss: 1.1575327789614676e-06
torch.Size([1, 15, 3, 96, 96]) torch.Size([5, 96, 96, 3])
Loss: 0.018930479884147644
torch.Size([1, 15, 3, 96, 96]) torch.Size([5, 96, 96

KeyboardInterrupt: 

: 